In [1]:
# Load the CSV
# import pandas as pd loads pandas so we can work with the data as a table.
# url holds the dataset address. pd.read_csv(url) downloads and reads the CSV directly into a DataFrame called df.
# df.shape gives (rows, columns). The file I checked had (5000, 9).
# df.head() shows the first five rows so you can see what one lead looks like. display() makes the table easier to read in Jupyter.


import pandas as pd 

url =("https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "course_lead_scoring_2026.csv")

df = pd.read_csv(url)

display(df.head())

print("row and coumn:", df.shape)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


row and coumn: (5000, 9)


In [2]:
# chech the target and missing value 
# df.info()	Shows each column’s data type and non-missing count. This helps us see which inputs are numbers and which are categories. The parentheses run the method.
# df["converted"].value_counts(dropna=False) Counts each target value and shows missing labels if there are any. We expect only 0 and 1.
# df.isna().sum().sort_values(ascending=False) Marks missing cells, counts them by column, and puts the columns with the most missing values first.

df.info()
print ("\n Target counts:")

print (df["converted"].value_counts(dropna=False))

print (df.isna().sum().sort_values(ascending=False))


<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB

 Target counts:
converted
1    2883
0    2117
Name: count, dtype: int64
annual_income               369
industry                    240
location                    208
employment_status           194
lead_source                 148
lead_sc

In [3]:
# Step 3 — Separate the inputs from the answer, then set aside a test set.
# Create X by removing the target and leaving out lead_score until we verify how and when it was calculated.
# Create y from converted, where 1 means converted and 0 means not converted.
# Split X and y together so each lead stays paired with its actual outcome.
# test_size=0.20 reserves 20% of the data for final testing.
# stratify=y keeps the 0/1 proportions similar in the training and test sets.
# random_state=42 gives the same split each time. The choice of 42 is arbitrary.
# y_train.mean() * 100 gives the percentage of training leads that converted.

from sklearn.model_selection import train_test_split
X = df.drop(columns=["converted", "lead_score"])
y = df["converted"]

# First split: reserve 20% of all leads for the final test.
X_remaining, X_test, y_remaining, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# Second split: take 25% of the remaining 80% for validation.

X_train, X_val, y_train, y_val = train_test_split(
    X_remaining, y_remaining, test_size=0.25, stratify = y_remaining,
    random_state = 42)

print("Training inputs:", X_train.shape)
print("Validation inputs:", X_val.shape)
print("Test inputs:", X_test.shape)



Training inputs: (3000, 7)
Validation inputs: (1000, 7)
Test inputs: (1000, 7)


In [4]:
# Check the numerical columns

In [5]:
# These three inputs contain numbers.
numeric_cols = [
    "annual_income",
    "number_of_courses_viewed",
    "interaction_count"
]

# Show the typical values and the range for each column.
display(X_train[numeric_cols].describe().round(2))

# Count how many training leads have a missing value in each column.
# display(X_train[numeric_cols].isna().sum())

,annual_income,number_of_courses_viewed,interaction_count
count,2780.00,3000.00,3000.00
mean,53890.96,2.09,4.48
std,21232.78,1.18,2.42
min,12000.00,0.00,0.00
25%,39788.00,1.00,3.00
50%,54069.00,2.00,4.00
75%,70750.50,3.00,6.00
max,109886.00,6.00,13.00


In [6]:
# Compare conversion rates by lead source

# Put each training lead's actual outcome next to their input columns.
train_eda = X_train.assign(converted=y_train)

# Group leads by source, keeping missing sources as their own group.
source_summary = (
    train_eda.groupby("lead_source", dropna=False)["converted"]
    .agg(leads="size", converted_leads="sum", conversion_rate="mean")
)

# Turn the conversion rate into a percentage and show the largest groups first.
source_summary["conversion_pct"] = (source_summary["conversion_rate"] * 100).round(2)
display(source_summary.sort_values("leads", ascending=False))

,leads,converted_leads,conversion_rate,conversion_pct
lead_source,,,,
organic_search,772,518,0.670984,67.10
paid_ads,698,300,0.429799,42.98
social_media,634,365,0.575710,57.57
referral,465,353,0.759140,75.91
events,351,148,0.421652,42.17
NaN,80,46,0.575000,57.50


In [7]:
# Pipeline runs preprocessing steps in order.
# ColumnTransformer applies the right steps to each group of columns.
# handle_unknown="ignore" lets the encoder process categories it hasn't seen before.

# This cell sets up the rules. X_train stays unchanged.
# Next, add Logistic Regression to the pipeline and call .fit(X_train, y_train).
# During fit, preprocessing learns the medians and categories from the 3,000 training leads.
# Logistic Regression then learns from the prepared inputs and their outcomes.


from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# List the columns that contain numbers.
numeric_cols = [
    "annual_income",
    "number_of_courses_viewed",
    "interaction_count"
]

# List the columns that contain category labels.
categorical_cols = [
    "lead_source",
    "industry",
    "employment_status",
    "location"
]

# For numbers: fill missing values first, then scale.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# For categories: keep missing values as their own category, then encode.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Apply the right preparation steps to each group of columns.
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

In [8]:
# Step 6 — Train your first Logistic Regression model

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Prepare the inputs first, then pass them to Logistic Regression.
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Learn the preparation rules and model weights from training data.
model.fit(X_train, y_train)

print("Logistic Regression training completed.")



Logistic Regression training completed.


In [9]:
# Compare Logistic Regression with a simple baseline to check how well it ranks the 1,000 validation leads, giving higher scores to those who actually converted.

#  model.predict_proba(X_val) prepares the validation data and predicts each lead’s probabilities using the trained model.
#  [:, 1] selects each lead’s probability of converting.
#  DummyClassifier(strategy="prior") gives every lead the same probability, based on the training conversion rate, to provide a baseline.
#  roc_auc_score(y_val, val_prob) measures how well the model ranks actual converters above non-converters, counting tied scores as half.
    
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

# Get the predicted probability of conversion (class 1) for each lead.
val_prob = model.predict_proba(X_val)[:, 1]

# Create a baseline that gives every lead the training conversion rate.
baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)
baseline_prob = baseline.predict_proba(X_val)[:, 1]

# Compare both sets of probabilities with the actual validation outcomes.
print(f"Baseline ROC AUC: {roc_auc_score(y_val, baseline_prob):.3f}")
print(f"Logistic Regression ROC AUC: {roc_auc_score(y_val, val_prob):.3f}")

Baseline ROC AUC: 0.500
Logistic Regression ROC AUC: 0.804


In [10]:
# Step 8 — See what happens when we change the cutoff. ROC AUC tells us how well the model ranks leads. The cutoff tells us which leads to flag for follow-up. For example, a lead scored 0.45 is flagged at a cutoff of 0.40, but isn’t flagged at 0.50.
# Leads flagged: How many of the 1,000 validation leads would we prioritise?
# Precision: Of the leads we flagged, how many actually converted?
# Recall: Of all the leads who converted, how many did we flag?
# F1: A single score that balances precision and recall.

from sklearn.metrics import precision_score, recall_score, f1_score

results = []

for cutoff in [0.30, 0.40, 0.50, 0.60, 0.70]:
    # Flag the lead (1) if their probability reaches the cutoff.
    val_pred = (val_prob >= cutoff).astype(int)

    # Compare the flags with the actual validation outcomes.
    results.append({
        "cutoff": cutoff,
        "leads_flagged": val_pred.sum(),
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred, zero_division=0),
        "f1": f1_score(y_val, val_pred, zero_division=0)
    })

display(pd.DataFrame(results).round(3))


,cutoff,leads_flagged,precision,recall,f1
0,0.3,808,0.667,0.936,0.779
1,0.4,709,0.725,0.892,0.800
2,0.5,611,0.753,0.799,0.775
3,0.6,497,0.793,0.684,0.734
4,0.7,376,0.835,0.545,0.660


In [11]:
# Cutoff 0.40 had the highest F1 among the tested cutoffs.
# It flagged 709 leads, with 72.5% precision and 89.2% recall.
# Keep this choice provisional until model comparison is complete.

In [15]:
# Step 9 — Tune C, the regularisation setting in Logistic Regression.

from sklearn.base import clone

# Try a small range of regularisation settings.
c_values = [0.001, 0.01, 0.1, 1, 10]
# c_values = [0.1, 0.01, 0.0001, 0.000001]
tuning_results = []
trained_models = {}

for c in c_values:
    # Create a fresh copy of the pipeline without learned values.
    candidate = clone(model)

    # Change C in the pipeline step named "classifier".
    candidate.set_params(classifier__C=c)

    # Learn preprocessing and model weights from training data only.
    candidate.fit(X_train, y_train)

    # Predict conversion probabilities for validation leads.
    candidate_prob = candidate.predict_proba(X_val)[:, 1]

    # Measure validation ranking quality.
    auc = roc_auc_score(y_val, candidate_prob)

    tuning_results.append({
        "C": c,
        "validation_auc": auc
    })

    # Keep each fitted pipeline so we can use the selected one later.
    trained_models[c] = candidate

# Show the settings with the highest validation AUC first.
tuning_table = pd.DataFrame(tuning_results).sort_values(
    "validation_auc", ascending=False
)

display(tuning_table.round(4))

,C,validation_auc
3,1.000,0.8043
4,10.000,0.8042
2,0.100,0.8039
1,0.010,0.8017
0,0.001,0.7969


In [16]:
# Keep the fitted Logistic Regression pipeline with C = 1.
best_lr = trained_models[1]

# Record its validation ranking score for model comparison.
best_lr_auc = roc_auc_score(
    y_val,
    best_lr.predict_proba(X_val)[:, 1]
)

print(f"Selected Logistic Regression validation AUC: {best_lr_auc:.4f}")

Selected Logistic Regression validation AUC: 0.8043


In [17]:
# Step 10 — Train Random Forest and compare it with Logistic Regression


from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# Copy the preparation rules without copying learned values.
rf_preprocessor = clone(preprocessor)

# Scaling is unnecessary for Random Forest.
# Keep numeric imputation, but skip StandardScaler.
rf_preprocessor.set_params(numeric__scaler="passthrough")

# Connect input preparation to a forest of 100 trees.
rf_model = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_leaf=10,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

# Learn preparation rules and trees from the same 3,000 training leads.
rf_model.fit(X_train, y_train)

# Get conversion probabilities for the same 1,000 validation leads.
rf_val_prob = rf_model.predict_proba(X_val)[:, 1]

# Compare ranking quality using the same metric.
rf_val_auc = roc_auc_score(y_val, rf_val_prob)

print(f"Logistic Regression validation AUC: {best_lr_auc:.4f}")
print(f"Random Forest validation AUC: {rf_val_auc:.4f}")

Logistic Regression validation AUC: 0.8043
Random Forest validation AUC: 0.8004


In [21]:
# Step 11 — Tune Random Forest

from sklearn.metrics import roc_auc_score

rf_results = []
best_rf = None
best_rf_auc = -1

for depth in [3, 5, 10]:
    for leaf_size in [5, 10, 20]:
        # Start with a fresh pipeline for each combination.
        candidate = clone(rf_model)
        candidate.set_params(
            classifier__max_depth=depth,
            classifier__min_samples_leaf=leaf_size
        )

        # Train on the training set and score on the validation set.
        candidate.fit(X_train, y_train)
        probabilities = candidate.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, probabilities)

        rf_results.append({
            "max_depth": depth,
            "min_samples_leaf": leaf_size,
            "validation_auc": auc
        })

        # Keep the fitted forest with the highest validation AUC.
        if auc > best_rf_auc:
            best_rf_auc = auc
            best_rf = candidate

display(
    pd.DataFrame(rf_results)
    .sort_values("validation_auc", ascending=False)
    .round(4)
)

print(f"Best Logistic Regression AUC: {best_lr_auc:.4f}")
print(f"Best Random Forest AUC: {best_rf_auc:.4f}")

,max_depth,min_samples_leaf,validation_auc
7,10,10,0.8014
8,10,20,0.8013
4,5,10,0.8004
3,5,5,0.7998
6,10,5,0.7997
5,5,20,0.7989
0,3,5,0.7957
1,3,10,0.7955
2,3,20,0.7954


Best Logistic Regression AUC: 0.8043
Best Random Forest AUC: 0.8014


In [22]:
# Step 12 — Confirm the 0.40 cutoff on the selected model.

from sklearn.metrics import precision_score, recall_score, f1_score

final_model = best_lr
final_cutoff = 0.40

# Get the selected model's conversion probabilities on validation data.
final_val_prob = final_model.predict_proba(X_val)[:, 1]

# Flag leads whose probability is at least 0.40.
final_val_pred = (final_val_prob >= final_cutoff).astype(int)

print("Leads flagged:", final_val_pred.sum())
print(f"Precision: {precision_score(y_val, final_val_pred):.3f}")
print(f"Recall:    {recall_score(y_val, final_val_pred):.3f}")
print(f"F1:        {f1_score(y_val, final_val_pred):.3f}")

Leads flagged: 709
Precision: 0.725
Recall:    0.892
F1:        0.800


In [24]:
# Step 13 — Evaluate the fixed model on the test set. Keep final_model and final_cutoff as they are. This is the final check on the 1,000 leads you set aside.

from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    precision_score, recall_score, f1_score
)

# Score each test lead using the already fitted model.
test_prob = final_model.predict_proba(X_test)[:, 1]

# Apply the 0.40 cutoff chosen on validation data.
test_pred = (test_prob >= final_cutoff).astype(int)

print(f"Test ROC AUC:   {roc_auc_score(y_test, test_prob):.3f}")
print(f"Test precision: {precision_score(y_test, test_pred, zero_division=0):.3f}")
print(f"Test recall:    {recall_score(y_test, test_pred, zero_division=0):.3f}")
print(f"Test F1:        {f1_score(y_test, test_pred, zero_division=0):.3f}")
print(f"Leads flagged:  {test_pred.sum()} of {len(y_test)}")
print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred, labels=[0, 1]))



Test ROC AUC:   0.776
Test precision: 0.689
Test recall:    0.872
Test F1:        0.770
Leads flagged:  730 of 1000
Confusion matrix:
[[196 227]
 [ 74 503]]
